<a href="https://colab.research.google.com/github/NguyenManhCuong1512/AI/blob/main/Stacking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris, load_breast_cancer, load_wine
from sklearn.model_selection import (
    train_test_split, cross_val_score, cross_val_predict,
    StratifiedKFold, GridSearchCV
)
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import (
    BaggingClassifier, RandomForestClassifier,
    AdaBoostClassifier, GradientBoostingClassifier,
    StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, classification_report, ConfusionMatrixDisplay
from xgboost import XGBClassifier  # pip install xgboost

# Dataset dùng xuyên suốt
bc   = load_breast_cancer()   # nhị phân, 569 mẫu, 30 đặc trưng
iris = load_iris()            # 3 lớp,   150 mẫu,  4 đặc trưng
wine = load_wine()            # 3 lớp,   178 mẫu, 13 đặc trưng

X_bc, y_bc     = bc.data,   bc.target
X_iris, y_iris = iris.data, iris.target
X_wine, y_wine = wine.data, wine.target

In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X_bc, y_bc, test_size=0.2, stratify=y_bc, random_state=42
)

base_models = {
    "lr":  Pipeline([("sc", StandardScaler()), ("clf", LogisticRegression(max_iter=1000, random_state=42))]),
    "rf":  RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "gb":  GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
    "knn": Pipeline([("sc", StandardScaler()), ("clf", KNeighborsClassifier(n_neighbors=7))]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Bước 1: OOF predictions trên tập train
oof_train = np.zeros((len(X_train), len(base_models)))
for j, (name, model) in enumerate(base_models.items()):
    oof_preds = cross_val_predict(model, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
    oof_train[:, j] = oof_preds
    print(f"OOF AUC ({name}): {roc_auc_score(y_train, oof_preds):.4f}")

# Bước 2: Predictions trên tập test (huấn luyện lại trên toàn bộ train)
oof_test = np.zeros((len(X_test), len(base_models)))
for j, (name, model) in enumerate(base_models.items()):
    model.fit(X_train, y_train)
    oof_test[:, j] = model.predict_proba(X_test)[:, 1]

# Bước 3: Meta-model
meta = LogisticRegression(max_iter=1000, random_state=42)
meta.fit(oof_train, y_train)
print(f"\nAUC — Stacking (thủ công): {roc_auc_score(y_test, meta.predict_proba(oof_test)[:, 1]):.4f}")

OOF AUC (lr): 0.9954
OOF AUC (rf): 0.9889
OOF AUC (gb): 0.9915
OOF AUC (knn): 0.9882

AUC — Stacking (thủ công): 0.9957


In [23]:
estimators = [
    ("lr",  Pipeline([("sc", StandardScaler()), ("clf", LogisticRegression(max_iter=1000, random_state=42))])),
    ("rf",  RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ("gb",  GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)),
    ("knn", Pipeline([("sc", StandardScaler()), ("clf", KNeighborsClassifier(n_neighbors=7))])),
]

stack_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000, random_state=42),
    cv=5,                       # OOF tự động
    stack_method="predict_proba",
    passthrough=False,          # True = thêm X gốc vào meta-features
    n_jobs=-1
)
stack_clf.fit(X_train, y_train)
print(f"AUC — StackingClassifier: {roc_auc_score(y_test, stack_clf.predict_proba(X_test)[:, 1]):.4f}")

AUC — StackingClassifier: 0.9957


In [24]:
all_models = {
    "Decision Tree":     DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest":     RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "XGBoost":           XGBClassifier(n_estimators=200, learning_rate=0.05, eval_metric="logloss", verbosity=0, random_state=42),
    "Stacking":          StackingClassifier(
                             estimators=estimators,
                             final_estimator=LogisticRegression(max_iter=1000, random_state=42),
                             cv=5, n_jobs=-1
                         ),
}

print(f"{'Model':<22} {'AUC mean':>10} {'AUC std':>10}")
print("-" * 44)
for name, model in all_models.items():
    scores = cross_val_score(model, X_bc, y_bc, cv=5, scoring="roc_auc", n_jobs=-1)
    print(f"{name:<22} {scores.mean():>10.4f} {scores.std():>10.4f}")

Model                    AUC mean    AUC std
--------------------------------------------
Decision Tree              0.9160     0.0223
Random Forest              0.9915     0.0066
XGBoost                    0.9935     0.0050
Stacking                   0.9962     0.0030
